In [2]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

## 📍 Lodaing **`all-MiniLM-L6-v2`** Model

In [3]:
model = SentenceTransformer(model_name_or_path="all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## 📍 Loading Datasets

In [4]:
jobs = pd.read_csv("Dataset/Jobs.csv")
companies = pd.read_csv("Dataset/Companies.csv")

## 🟢 filtering job descriptions to reduce the number of irrelevant ones

In [15]:
filterd_data = jobs[(jobs["Job Title"] == "Teacher") & (jobs["Work Type"] == "Intern") & ((jobs["Preference"] == "Male") | (jobs["Preference"] == "Both")) ]
filterd_companies = companies[companies["Job Id"].isin(filterd_data["Job Id"].to_list())]

## 🟢 Employer cv Description 

In [16]:
employer_description = 'master teacher highlights home schooling knowledge calm and patient certified in early childhood education head start programs strong communicator toddler and preschool curricula classroom management classroom management skills i have a lot patience  i communicate really well with children  i can usually calm a stressful situation quickly  education early childhood education      san jacinto college     city   state   usa coursework in child nutritionchild abuse awareness trainingcoursework in emergency preparednesscoursework in behavior managementemphasis in child development high school diploma   general      beaumont high school     city   state   usa emphasis in child development experience master teacher sep      to apr       company name     city   state conducted small group and individual classroom activities based on differentiated learning needs encouraged students to be understanding of and helpful to others observed students to supply teachers with feedback regarding potential learning blocks and opportunities for support physically and verbally interacted with students throughout the day to keep them engaged supported students in developing strategies for individual needs and classroom group dynamics organized field trips to local parks  fire stations and zoos helped prepare daily lesson plans for activities and lessons implemented emergent curriculum to encourage student participation supplied one on one attention to each student  while maintaining overall focus on the entire group wrote daily and weekly lesson plans administered minor first aid to injured students applied the positive reinforcement method to redirect negative behaviors promoted language development skills through reading and storytelling  implemented family style meals  conducting parent teacher conferences  and kept personal profiles of each child  teachers associate aug      to nov       company name     city   state promoted language development skills through reading and storytelling applied the positive reinforcement method to redirect negative behaviors conducted small group and individual classroom activities based on differentiated learning needs assisted     children per station during small group learning periods  organized field trips to encouraged students to be understanding of and helpful to others observed students to supply teachers with feedback regarding potential learning blocks and opportunities for support physically and verbally interacted with students throughout the day to keep them engaged helped prepare daily lesson plans for activities and lessons implemented emergent curriculum supplied one on one attention to each student  while maintaining overall focus on the entire group physically and verbally interacted with students throughout the day to keep them engaged communicated effectively with educators from various grade levels wrote daily and weekly lesson plans administered minor first aid to injured students to encourage student participation organized field trips to local parks  fire stations and zoos maintained daily records of children s individual activities  behaviors  meals and naps  established a safe play environment for the children distributed quarterly educational assessments  similar to report cards  to each parent developed professional relationships with parents  teachers  directors and therapists observed children for signs of illness  injury  emotional disturbance  learning disorders and speech problems maintained daily records of children s individual activities  behaviors  meals and naps  teacher aide aug      to mar       company name     city   state promoted language development skills through reading and storytelling applied the positive reinforcement method to redirect negative behaviors conducted small group and individual classroom activities based on differentiated learning needs  organized field trips to local parks  fire stations and zoos  assisted     children per station during small group learning periods encouraged students to be understanding of and helpful to others observed students to supply teachers with feedback regarding potential learning blocks and opportunities for support physically and verbally interacted with students throughout the day to keep them engaged supported students in developing strategies for individual needs and classroom group dynamics helped prepare daily lesson plans for activities and lessons  implemented emergent curriculum to encourage student participation  supplied one on one attention to each student  while maintaining overall focus on the entire group communicated effectively with educators from various grade levels wrote daily and weekly lesson plans administered minor first aid to injured students maintained daily records of children s individual activities  behaviors  meals and naps established a safe play environment for the children created and implemented developmentally appropriate curriculum that addressed all learning styles distributed quarterly educational assessments  similar to report cards  to each parent maintained a child friendly environment with access to outdoor activities completed all required documentation for the national head start program  extra curricular activities  i ran the nursery at my church for a yea and then taught sunday school for the older kids for another two years   i also helped run an after school  get your homework done here  program the community started where kids that didn t have help at home could come to us and get help with their homework or for the younger children the parents could enroll them and they would come daily after school  summary talented early education professional with diverse experience in planning and implementing various activities for promoting physical  social  emotional and intellectual growth of children  '

## ✔ Embedding Jobs Descriptions & Employer Description

In [17]:
jd_embedding = model.encode([employer_description])
resume_embeddings = model.encode(filterd_data["Job Description"].to_list())

## ✔ Calculate Cosine Similarity
* This returns a score between **0 and 1** representing semantic closeness

In [18]:
similarity_scores = cosine_similarity(jd_embedding, resume_embeddings)[0]

## 🚩 Rank and Format the Results

In [22]:
results = []
for i, score in enumerate(similarity_scores):
    job_data = filterd_data.iloc[i]
    company_data = filterd_companies.iloc[i]

    results.append({
        "Job Description": job_data["Job Description"],
        "Salary Range":job_data['Salary Range'],
        "Benefits":job_data['Benefits'],
        "skills":job_data['skills'],
        "Contact Person":company_data['Contact Person'],
        "Contact":company_data['Contact'],
        "Responsibilities":job_data['Responsibilities'],

        "Company" :company_data["Company"],
        "Company Profile" :company_data["Company Profile"],
        "Company Size" :company_data["Company Size"],
        "location" :company_data["location"],
        "Country" :company_data["Country"],
        "latitude" :company_data["latitude"],
        "longitude" :company_data["longitude"],
        "Date":job_data['Date'],
        "Preference": job_data["Preference"],
        "Match_Score": round(score * 100, 2), 
    })


## 🚩 Convert to Pandas DataFrame for a clean, analytical output

In [23]:
pd.DataFrame(results).sort_values(by="Match_Score", ascending=False)

,Job Description,Salary Range,Benefits,skills,Contact Person,Contact,Responsibilities,Company,Company Profile,Company Size,location,Country,latitude,longitude,Date,Preference,Match_Score
7,A Classroom Teacher educates students in a spe...,$55K-$127K,"{'Health Insurance, Retirement Plans, Paid Tim...",Teaching pedagogy Classroom management Curricu...,Gerald Wilson,(882)564-5985x882,"Plan and deliver engaging lessons, adapting te...",Larsen & Toubro Infotech,"{""Sector"":""Information Technology"",""Industry"":...",91416,Khartoum,Sudan,12.862800,30.217600,2023-01-22,Both,47.680000
1339,A Classroom Teacher educates students in a spe...,$62K-$106K,"{'Flexible Spending Accounts (FSAs), Relocatio...",Teaching pedagogy Classroom management Curricu...,Jessica Thomas,295-256-2945x7823,"Plan and deliver engaging lessons, adapting te...",Oracle,"{""Sector"":""Technology"",""Industry"":""Computer So...",128448,Vilnius,Lithuania,55.169400,23.881300,2022-09-25,Male,47.680000
1353,A Classroom Teacher educates students in a spe...,$55K-$80K,"{'Flexible Spending Accounts (FSAs), Relocatio...",Teaching pedagogy Classroom management Curricu...,William Jones,(397)550-3118x4499,"Plan and deliver engaging lessons, adapting te...",Occidental Petroleum,"{""Sector"":""Energy"",""Industry"":""Mining, Crude-O...",78332,Bucharest,Romania,45.943200,24.966800,2023-04-10,Male,47.680000
0,A Classroom Teacher educates students in a spe...,$56K-$87K,"{'Health Insurance, Retirement Plans, Paid Tim...",Teaching pedagogy Classroom management Curricu...,Michael Kaufman,001-938-901-1119x729,"Plan and deliver engaging lessons, adapting te...",Centrica,"{""Sector"":""Utilities"",""Industry"":""Utilities"",""...",71603,Wellington,New Zealand,-40.900600,174.886000,2022-01-06,Male,47.680000
14,A Classroom Teacher educates students in a spe...,$58K-$95K,"{'Casual Dress Code, Social and Recreational A...",Teaching pedagogy Classroom management Curricu...,Earl Tran,(915)581-9033x04731,"Plan and deliver engaging lessons, adapting te...",General Electric,"{""Sector"":""Conglomerate"",""Industry"":""Industria...",103590,Mbabane,Eswatini,26.522500,31.465900,2022-05-24,Male,47.680000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
816,A Subject Matter Expert possesses deep knowled...,$63K-$80K,"{'Casual Dress Code, Social and Recreational A...",Expertise in a specific subject area Knowledge...,Larry Espinoza,318-508-9360,Specialize in a particular subject area and de...,Glenmark Pharmaceuticals,"{""Sector"":""Pharmaceuticals"",""Industry"":""Pharma...",95343,Doha,Qatar,25.354800,51.183900,2022-02-23,Both,17.190001
817,A Subject Matter Expert possesses deep knowled...,$60K-$119K,"{'Life and Disability Insurance, Stock Options...",Expertise in a specific subject area Knowledge...,Amber Johnson,514-591-8720,Specialize in a particular subject area and de...,PPL,"{""Sector"":""Energy"",""Industry"":""Utilities: Gas ...",38403,Pristina,Kosovo,42.602600,20.903000,2022-09-13,Male,17.190001
866,A Subject Matter Expert possesses deep knowled...,$60K-$109K,"{'Childcare Assistance, Paid Time Off (PTO), R...",Expertise in a specific subject area Knowledge...,Angela Warren,4618339234,Specialize in a particular subject area and de...,Petronet LNG,"{""Sector"":""Oil & Gas"",""Industry"":""Oil and Gas""...",116090,AsunciÃ³n,Paraguay,-23.442503,-58.443832,2022-03-17,Both,17.190001
869,A Subject Matter Expert possesses deep knowled...,$58K-$90K,"{'Employee Assistance Programs (EAP), Tuition ...",Expertise in a specific subject area Knowledge...,Amanda Dickerson,+1-807-672-9922x2482,Specialize in a particular subject area and de...,CommScope Holding,"{""Sector"":""Telecommunications"",""Industry"":""Net...",95908,Dhaka,Bangladesh,23.685000,90.356300,2022-04-18,Both,17.190001


## 📍 Functions

In [ ]:
def SimilarityCVwithJobs(Jobs:pd.DataFrame,Companies:pd.DataFrame,Employer_Desc:str):
    model = SentenceTransformer(model_name_or_path="all-MiniLM-L6-v2")
    filterd_Jobs = Jobs[(Jobs["Job Title"] == "Teacher") & (Jobs["Work Type"] == "Intern") & (Jobs["Preference"] == "Male" or Jobs["Preference"] == "Both" ) ]
    filterd_companies = Companies[Companies["Job Id"].isin(filterd_Jobs["Job Id"].to_list())]

    jd_embedding = model.encode([Employer_Desc])
    resume_embeddings = model.encode(filterd_Jobs["Job Description"].to_list())
    
    return cosine_similarity(jd_embedding, resume_embeddings)[0] , filterd_Jobs , filterd_companies

In [ ]:
def GetTopJobs(Similarity_Scores:np.ndarray,Filterd_Jobs:pd.DataFrame,Filterd_Companies:pd.DataFrame):
    results = []
    for i, score in enumerate(Similarity_Scores):
        job_data = Filterd_Jobs.iloc[i]
        company_data = Filterd_Companies.iloc[i]

        results.append({
            "Job Description": job_data["Job Description"],
            "Salary Range":job_data['Salary Range'],
            "Benefits":job_data['Benefits'],
            "skills":job_data['skills'],
            "Contact Person":company_data['Contact Person'],
            "Contact":company_data['Contact'],
            "Responsibilities":job_data['Responsibilities'],

            "Company" :company_data["Company"],
            "Company Profile" :company_data["Company Profile"],
            "Company Size" :company_data["Company Size"],
            "location" :company_data["location"],
            "Country" :company_data["Country"],
            "latitude" :company_data["latitude"],
            "longitude" :company_data["longitude"],
            "Date":job_data['Date'],
            "Match_Score": round(score * 100, 2), 
        })
    
    return pd.DataFrame(results).sort_values(by="Match_Score", ascending=False)


---

<h1 align="center">✅ The End of Model Notebook</h1>


---